# PhenoAssistant on Microsoft Agent Framework

CPU-only migration of the analysis section of `demo.ipynb`.

This notebook reuses the tracked dataset
`results/demo/potato_phenotypes.csv`. It does not rerun instance
segmentation and does not modify the original AutoGen notebook.


In [ ]:
from __future__ import annotations

import inspect
import os
from pathlib import Path
from typing import Any

from phenoassistant_maf import (
    OpenRouterSettings,
    build_application,
    create_chat_client,
    run_application,
)

ROOT = Path.cwd()
DATA_PATH = ROOT / "results/demo/potato_phenotypes.csv"
OUTPUT_DIR = ROOT / "results/maf_demo"
MANUAL_PLOT = OUTPUT_DIR / "potato_manual.png"
ALGORITHM_PLOT = OUTPUT_DIR / "potato_algorithm.png"

assert DATA_PATH.is_file()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATASET_ROWS=", sum(1 for _ in DATA_PATH.open()) - 1)


In [ ]:
if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before running this cell.")

os.environ.setdefault("OPENROUTER_MODEL", "openrouter/free")

settings = OpenRouterSettings.from_env()
client = create_chat_client(settings)


def forbidden_calculator(
    a: int,
    b: int,
    operator: str,
) -> int:
    raise AssertionError(
        "The demo Manager selected calculator unexpectedly."
    )


def forbidden_anova(
    data_path: str,
    descriptor: str,
    within_subject_factor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError(
        "The demo Manager selected ANOVA unexpectedly."
    )


def forbidden_tukey(
    data_path: str,
    descriptor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError(
        "The demo Manager selected Tukey unexpectedly."
    )


application = build_application(
    client=client,
    data_path=str(DATA_PATH),
    calculator_callable=forbidden_calculator,
    anova_callable=forbidden_anova,
    tukey_callable=forbidden_tukey,
    first_plot_path=str(MANUAL_PLOT),
    second_plot_path=str(ALGORITHM_PLOT),
)

print("MODEL=", settings.model)
print("TOOLS=", application.registry.names)


In [ ]:
def response_text(response: Any) -> str:
    messages = getattr(response, "messages", None)
    if messages:
        text = getattr(messages[-1], "text", None)
        if isinstance(text, str):
            return text.strip()
    return str(response).strip()


In [ ]:
regression_response = await run_application(
    application,
    '''
    Using the trusted potato dataset, compare manual_leaf_area and
    projected_leaf_area as predictors of manual_dried_weight. Call
    compare_linear_relationships exactly once. Report both equations,
    both Pearson correlations, and which relationship is stronger using
    only returned tool evidence.
    ''',
)
print(response_text(regression_response))


In [ ]:
maximum_response = await run_application(
    application,
    '''
    Find the maximum manual_leaf_area where Variety equals Desiree.
    Call query_csv_statistic exactly once using the maximum operation.
    Report the value and matching row count using only tool evidence.
    ''',
)
print(response_text(maximum_response))


In [ ]:
mean_response = await run_application(
    application,
    '''
    Find the mean manual_leaf_area where Variety equals Desiree.
    Call query_csv_statistic exactly once using the mean operation.
    Report the value and matching row count using only tool evidence.
    ''',
)
print(response_text(mean_response))


In [ ]:
assert MANUAL_PLOT.is_file()
assert ALGORITHM_PLOT.is_file()
assert MANUAL_PLOT.stat().st_size > 0
assert ALGORITHM_PLOT.stat().st_size > 0
print("MAF_DEMO_WORKFLOW_COMPLETE")

async def close_provider(value: Any) -> None:
    for candidate in (
        value,
        getattr(value, "client", None),
        getattr(value, "_client", None),
        getattr(value, "_openai_client", None),
    ):
        if candidate is None:
            continue
        for name in ("aclose", "close"):
            method = getattr(candidate, name, None)
            if callable(method):
                result = method()
                if inspect.isawaitable(result):
                    await result
                return

await close_provider(client)
print("OPENROUTER_CLIENT_CLOSED")
